In [6]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Delta-Iceberg-MinIO")
    .master("local[*]")
    .config("spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension,"
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "hadoop")
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/iceberg/")
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true")
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1")
    .config("spark.sql.catalog.iceberg.s3.access-key-id", "matrix")
    .config("spark.sql.catalog.iceberg.s3.secret-access-key", "matrix123")
    .getOrCreate()
)


In [7]:
hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [8]:
import os

for root, dirs, files in os.walk("/home/jovyan"):
    for f in files:
        if "transaction" in f.lower():
            print(os.path.join(root, f))

/home/jovyan/transactions_raw.csv


In [10]:
CSV_PATH = "/home/jovyan/transactions_raw.csv"

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CSV_PATH)
)

total = raw_df.count()
print(f"Total: {total:,}")
raw_df.printSchema()
raw_df.show(5, truncate=False)

Total: 52,100
root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- dt: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)

+-----+-----------+----------+------+--------+---------+
|id   |customer_id|dt        |amount|currency|status   |
+-----+-----------+----------+------+--------+---------+
|39329|247        |2025-03-26|163.99|AZN     |completed|
|41803|685        |2025-03-24|410.52|EUR     |completed|
|7150 |385        |2025-03-02|387.52|AZN     |completed|
|20125|726        |2025-02-15|221.73|EUR     |pending  |
|40951|219        |2025-02-05|581.89|AZN     |completed|
+-----+-----------+----------+------+--------+---------+
only showing top 5 rows



In [11]:
# Data Quality Checks
null_amount     = raw_df.filter(F.col("amount").isNull()).count()
null_customer   = raw_df.filter(F.col("customer_id").isNull()).count()
negative_amount = raw_df.filter(F.col("amount") < 0).count()
duplicate_ids   = raw_df.groupBy("id").count().filter(F.col("count") > 1).count()

print(f"NULL amount     : {null_amount}")
print(f"NULL customer_id: {null_customer}")
print(f"Mənfi amount    : {negative_amount}")
print(f"Duplicate id    : {duplicate_ids}\n")

print("Currency bölgüsü:")
raw_df.groupBy("currency").count().show()

print("Status bölgüsü:")
raw_df.groupBy("status").count().show()

NULL amount     : 500
NULL customer_id: 200
Mənfi amount    : 600
Duplicate id    : 800

Currency bölgüsü:
+--------+-----+
|currency|count|
+--------+-----+
|     EUR| 7921|
|     AZN|31249|
|     USD|12930|
+--------+-----+

Status bölgüsü:
+---------+-----+
|   status|count|
+---------+-----+
|completed|43855|
|   failed| 2633|
|  pending| 5612|
+---------+-----+



In [12]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bronze")

DELTA_BRONZE   = "s3a://warehouse/delta/bronze/transactions"
ICEBERG_BRONZE = "iceberg.bronze.transactions"

raw_df.write.format("delta").mode("overwrite").save(DELTA_BRONZE)
print(f"Delta Bronze yazıldı")

spark.sql(f"DROP TABLE IF EXISTS {ICEBERG_BRONZE}")
raw_df.writeTo(ICEBERG_BRONZE).createOrReplace()
print(f"Iceberg Bronze yazıldı")

Delta Bronze yazıldı
Iceberg Bronze yazıldı


In [13]:
DELTA_BRONZE = "s3a://warehouse/delta/bronze/transactions"

# Bronze məlumatının oxunması
bronze_df = spark.read.format("delta").load(DELTA_BRONZE)
total = bronze_df.count()

print(f"Ümumi Bronze sətir sayı: {total:,}")

Ümumi Bronze sətir sayı: 52,100


In [15]:
from pyspark.sql import functions as F

DELTA_BRONZE = "s3a://warehouse/delta/bronze/transactions"
DELTA_SILVER = "s3a://warehouse/delta/silver/transactions"
ICEBERG_SILVER = "iceberg.silver.transactions"

bronze_df = spark.read.format("delta").load(DELTA_BRONZE)
total = bronze_df.count()

silver_df = (
    bronze_df
    .filter(F.col("id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("dt").isNotNull())
    .filter(F.col("amount").isNotNull())  # 👈 Amount NULL olanlar buradan silinir
    .dropDuplicates(["id"])
    .orderBy("id")
)

silver_count = silver_df.count()
print(f"Bronze: {total:,} | Silver: {silver_count:,} | Silinən: {total - silver_count:,}")

silver_df.write.format("delta").mode("overwrite").save(DELTA_SILVER)

spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.silver")
spark.sql(f"DROP TABLE IF EXISTS {ICEBERG_SILVER}")
silver_df.writeTo(ICEBERG_SILVER).createOrReplace()

print("Silver qatına (Delta və Iceberg) uğurla yazıldı!")

Bronze: 52,100 | Silver: 50,600 | Silinən: 1,500
Silver qatına (Delta və Iceberg) uğurla yazıldı!


In [21]:
raw_df.show(10, truncate=False)

+-----+-----------+----------+-------+--------+---------+
|id   |customer_id|dt        |amount |currency|status   |
+-----+-----------+----------+-------+--------+---------+
|39329|247        |2025-03-26|163.99 |AZN     |completed|
|41803|685        |2025-03-24|410.52 |EUR     |completed|
|7150 |385        |2025-03-02|387.52 |AZN     |completed|
|20125|726        |2025-02-15|221.73 |EUR     |pending  |
|40951|219        |2025-02-05|581.89 |AZN     |completed|
|50599|418        |2025-01-24|-130.86|AZN     |pending  |
|12880|367        |2025-02-14|14.17  |AZN     |completed|
|660  |659        |2025-01-16|177.07 |AZN     |completed|
|20765|592        |2025-03-28|136.41 |AZN     |completed|
|26776|827        |2025-02-18|260.71 |AZN     |pending  |
+-----+-----------+----------+-------+--------+---------+
only showing top 10 rows



In [19]:
delta_silver_df = spark.read.format("delta").load(DELTA_SILVER)
delta_silver_df.show(10, truncate=False)

+---+-----------+----------+------+--------+---------+
|id |customer_id|dt        |amount|currency|status   |
+---+-----------+----------+------+--------+---------+
|1  |642        |2025-02-21|584.01|EUR     |completed|
|2  |934        |2025-01-15|215.26|AZN     |completed|
|3  |711        |2025-03-13|124.14|AZN     |completed|
|4  |574        |2025-03-02|322.37|USD     |completed|
|5  |596        |2025-01-21|210.36|AZN     |completed|
|6  |572        |2025-03-24|280.32|EUR     |pending  |
|7  |308        |2025-03-28|423.78|USD     |completed|
|8  |670        |2025-03-16|194.36|AZN     |pending  |
|9  |147        |2025-03-16|164.8 |AZN     |completed|
|10 |672        |2025-03-29|85.95 |USD     |completed|
+---+-----------+----------+------+--------+---------+
only showing top 10 rows



In [22]:
from pyspark.sql import functions as F
from pyspark.sql.functions import when, trim

DELTA_SILVER = "s3a://warehouse/delta/silver/transactions"
ICEBERG_SILVER = "iceberg.silver.transactions"

# 1. Cədvəldəki bütün string tipli sütunları avtomatik tapırıq
string_cols = [field.name for field in bronze_df.schema.fields if field.dataType.simpleString() == "string"]

# 2. Həmin sütunlardakı boşluqları və boş sətirləri NULL-a çeviririk
cleaned_bronze_df = bronze_df
for c in string_cols:
    cleaned_bronze_df = cleaned_bronze_df.withColumn(
        c, when(trim(F.col(c)) == "", None).otherwise(F.col(c))
    )

silver_df = (
    cleaned_bronze_df
    .filter(F.col("id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("dt").isNotNull())
    .filter(F.col("amount").isNotNull())
    .dropDuplicates(["id"])
    .orderBy("id")
)

silver_count = silver_df.count()
print(f"Bronze: {total:,} | Silver: {silver_count:,} | Silinən: {total - silver_count:,}")

# 4. Delta Silver-ə yaz
silver_df.write.format("delta").mode("overwrite").save(DELTA_SILVER)

# 5. Iceberg Silver-ə yaz
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.silver")
spark.sql(f"DROP TABLE IF EXISTS {ICEBERG_SILVER}")
silver_df.writeTo(ICEBERG_SILVER).createOrReplace()

print("Bütün mətn sütunları təmizlənərək Silver qatına uğurla yazıldı!")

Bronze: 52,100 | Silver: 50,600 | Silinən: 1,500
Bütün mətn sütunları təmizlənərək Silver qatına uğurla yazıldı!


In [23]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.gold")

DELTA_GOLD = "s3a://warehouse/delta/gold/daily_summary"
ICEBERG_GOLD = "iceberg.gold.daily_summary"

gold_df = (
    silver_df
    .groupBy("dt", "currency")
    .agg(
        F.count("id").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.sum(F.when(F.col("status") == "completed", 1).otherwise(0)).alias("completed"),
        F.sum(F.when(F.col("status") == "failed", 1).otherwise(0)).alias("failed"),
        F.sum(F.when(F.col("status") == "pending", 1).otherwise(0)).alias("pending"),
    )
    .orderBy("dt", "currency")
)

gold_df.show(10)

gold_df.write.format("delta").mode("overwrite").save(DELTA_GOLD)

spark.sql(f"DROP TABLE IF EXISTS {ICEBERG_GOLD}")
gold_df.writeTo(ICEBERG_GOLD).createOrReplace()

print("Gold yazıldı")

+----------+--------+-----------------+------------+----------+----------------+---------+------+-------+
|        dt|currency|transaction_count|total_amount|avg_amount|unique_customers|completed|failed|pending|
+----------+--------+-----------------+------------+----------+----------------+---------+------+-------+
|2025-01-01|     AZN|              338|    79560.51|    235.39|             275|      285|    13|     40|
|2025-01-01|     EUR|               66|    14668.71|    222.25|              63|       56|     3|      7|
|2025-01-01|     USD|              137|    30555.19|    223.03|             132|      114|     8|     15|
|2025-01-02|     AZN|              335|     77816.7|    232.29|             283|      292|    10|     33|
|2025-01-02|     EUR|               82|    19577.05|    238.74|              79|       72|     4|      6|
|2025-01-02|     USD|              139|    27819.18|    200.14|             128|      119|     6|     14|
|2025-01-03|     AZN|              354|    813

In [25]:
from delta.tables import DeltaTable
from pyspark.sql.types import DateType, IntegerType

new_rows = spark.createDataFrame([
    (99001, 101, "2025-04-01", 500.00, "AZN", "completed"),
    (99002, 102, "2025-04-01", 750.50, "USD", "pending"),
], ["id", "customer_id", "dt", "amount", "currency", "status"])

new_rows = (new_rows
    .withColumn("id", F.col("id").cast(IntegerType()))
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("dt", F.col("dt").cast(DateType()))
)

new_rows.write.format("delta").mode("append").save(DELTA_SILVER)
new_rows.writeTo(ICEBERG_SILVER).append()

print(f" INSERT — Delta: {spark.read.format('delta').load(DELTA_SILVER).count()}")
print(f" INSERT — Iceberg: {spark.table(ICEBERG_SILVER).count()}")
spark.read.format("delta").load(DELTA_SILVER).filter(F.col("id").isin([99001, 99002])).show()

 INSERT — Delta: 50602
 INSERT — Iceberg: 50602
+-----+-----------+----------+------+--------+---------+
|   id|customer_id|        dt|amount|currency|   status|
+-----+-----------+----------+------+--------+---------+
|99001|        101|2025-04-01| 500.0|     AZN|completed|
|99002|        102|2025-04-01| 750.5|     USD|  pending|
+-----+-----------+----------+------+--------+---------+



In [37]:
delta_table = DeltaTable.forPath(spark, DELTA_SILVER)

# Delta UPDATE
delta_table.update(
    condition = F.col("id") == 99001,
    set = {"status": F.lit("failed"), "amount": F.lit(999.99)}
)

# Iceberg UPDATE
spark.sql(f"UPDATE {ICEBERG_SILVER} SET status='failed', amount=999.99 WHERE id=99001")

print("UPDATE — id=99001 → status:failed, amount:999.99")
spark.read.format("delta").load(DELTA_SILVER).filter(F.col("id") == 99001).show()
spark.table(ICEBERG_SILVER).filter(F.col("id") == 99001).show()

UPDATE — id=99001 → status:failed, amount:999.99
+-----------+-----+----------+------+--------+------+-------------+
|customer_id|   id|        dt|amount|currency|status|loyalty_score|
+-----------+-----+----------+------+--------+------+-------------+
|        101|99001|2025-04-01|999.99|     AZN|failed|       BRONZE|
+-----------+-----+----------+------+--------+------+-------------+

+-----+-----------+----------+------+--------+------+-------------+
|   id|customer_id|        dt|amount|currency|status|loyalty_score|
+-----+-----------+----------+------+--------+------+-------------+
|99001|        101|2025-04-01|999.99|     AZN|failed|       BRONZE|
+-----+-----------+----------+------+--------+------+-------------+



In [38]:
# Delta DELETE
delta_table.delete(F.col("id") == 99002)

# Iceberg DELETE
spark.sql(f"DELETE FROM {ICEBERG_SILVER} WHERE id=99002")

print("DELETE — id=99002 silindi")
print(f"Delta count: {spark.read.format('delta').load(DELTA_SILVER).count()}")
print(f"Iceberg count: {spark.table(ICEBERG_SILVER).count()}")

DELETE — id=99002 silindi
Delta count: 50602
Iceberg count: 50602


In [28]:
upsert_df = spark.createDataFrame([
    (99001, 101, "2025-04-01", 1111.11, "AZN", "completed"),  # mövcuddur UPDATE etsin
    (99003, 103, "2025-04-02", 300.00,  "EUR", "completed"),  # yenidir INSERT etsin
], ["id", "customer_id", "dt", "amount", "currency", "status"])

upsert_df = (upsert_df
    .withColumn("id",          F.col("id").cast(IntegerType()))
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("dt",          F.col("dt").cast(DateType()))
)

# Delta MERGE
delta_table.alias("t").merge(
    upsert_df.alias("s"), "t.id = s.id"
).whenMatchedUpdate(set={
    "amount": "s.amount", "status": "s.status"
}).whenNotMatchedInsert(values={
    "id": "s.id", "customer_id": "s.customer_id", "dt": "s.dt",
    "amount": "s.amount", "currency": "s.currency", "status": "s.status"
}).execute()

# Iceberg MERGE
upsert_df.createOrReplaceTempView("upsert_src")
spark.sql(f"""
    MERGE INTO {ICEBERG_SILVER} AS t
    USING upsert_src AS s ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET t.amount = s.amount, t.status = s.status
    WHEN NOT MATCHED THEN INSERT *
""")

print(" MERGE — 99001 update, 99003 insert")
spark.read.format("delta").load(DELTA_SILVER).filter(F.col("id").isin([99001, 99003])).show()
spark.table(ICEBERG_SILVER).filter(F.col("id").isin([99001, 99003])).show()

 MERGE — 99001 update, 99003 insert
+-----+-----------+----------+-------+--------+---------+
|   id|customer_id|        dt| amount|currency|   status|
+-----+-----------+----------+-------+--------+---------+
|99001|        101|2025-04-01|1111.11|     AZN|completed|
|99003|        103|2025-04-02|  300.0|     EUR|completed|
+-----+-----------+----------+-------+--------+---------+

+-----+-----------+----------+-------+--------+---------+
|   id|customer_id|        dt| amount|currency|   status|
+-----+-----------+----------+-------+--------+---------+
|99001|        101|2025-04-01|1111.11|     AZN|completed|
|99003|        103|2025-04-02|  300.0|     EUR|completed|
+-----+-----------+----------+-------+--------+---------+



In [29]:
delta_table = DeltaTable.forPath(spark, DELTA_SILVER)

delta_table.history().select(
    "version", "timestamp", "operation"
).show(truncate=False)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|8      |2026-09-06 12:35:42|MERGE    |
|7      |2026-09-06 12:32:09|DELETE   |
|6      |2026-09-06 12:31:05|UPDATE   |
|5      |2026-09-06 12:30:28|WRITE    |
|4      |2026-09-06 12:18:11|WRITE    |
|3      |2026-09-06 11:56:08|WRITE    |
|2      |2026-09-06 11:54:22|WRITE    |
|1      |2026-09-01 16:58:47|WRITE    |
|0      |2026-09-01 16:52:18|WRITE    |
+-------+-------------------+---------+



In [31]:
# Delta Time Travel — version 0 (ilk yazılan raw data)
print("Delta v0 (ilk WRITE):")
spark.read.format("delta").option("versionAsOf", 0).load(DELTA_SILVER).show(5)

# Delta Time Travel — version 6 (UPDATE əməliyyatından əvvəl)
print("Delta v5 (UPDATE-dən əvvəl):")
spark.read.format("delta").option("versionAsOf", 5).load(DELTA_SILVER).filter(F.col("id") == 99001).show()

# Delta Time Travel — timestamp ilə
print("Delta timestamp ilə:")
spark.read.format("delta").option("timestampAsOf", "2026-09-06 12:18:00").load(DELTA_SILVER).count()

# Iceberg Time Travel — SQL ilə etdim error olurdu deyə
first_snapshot = snapshots[0]["snapshot_id"]
last_snapshot  = snapshots[-1]["snapshot_id"]

print(f"İlk snapshot ({first_snapshot}):")
spark.sql(f"""
    SELECT * FROM {ICEBERG_SILVER}
    VERSION AS OF {first_snapshot}
""").show(5)

print(f"Son snapshot ({last_snapshot}):")
spark.sql(f"""
    SELECT * FROM {ICEBERG_SILVER}
    VERSION AS OF {last_snapshot}
""").show(5)

Delta v0 (ilk WRITE):
+---+-----------+----------+------+--------+---------+
| id|customer_id|        dt|amount|currency|   status|
+---+-----------+----------+------+--------+---------+
|  1|        642|2025-02-21|584.01|     EUR|completed|
|  2|        934|2025-01-15|215.26|     AZN|completed|
|  3|        711|2025-03-13|124.14|     AZN|completed|
|  4|        574|2025-03-02|322.37|     USD|completed|
|  5|        596|2025-01-21|210.36|     AZN|completed|
+---+-----------+----------+------+--------+---------+
only showing top 5 rows

Delta v5 (UPDATE-dən əvvəl):
+-----+-----------+----------+------+--------+---------+
|   id|customer_id|        dt|amount|currency|   status|
+-----+-----------+----------+------+--------+---------+
|99001|        101|2025-04-01| 500.0|     AZN|completed|
+-----+-----------+----------+------+--------+---------+

Delta timestamp ilə:
İlk snapshot (5166888719585740639):
+---+-----------+----------+------+--------+---------+
| id|customer_id|        dt|amo

## Yekun Hesabat

### 1. Medallion Architecture
- **Bronze**: Raw data dəyişdirilmədən saxlanılır (52,100 sətir)
- **Silver**: Təmizlənmiş data — NULL, duplicate silindi (50,600 sətir)
- **Gold**: Biznes analizi üçün gündəlik/currency summary

### 2. Data Quality Nəticələri
| Problem | Say |
|---|---|
| NULL amount | 500 |
| NULL customer_id | 200 |
| Mənfi amount | 600 |
| Duplicate id | 800 |
| **Cəmi silindi** | **1,500** |

### 3. Delta vs Iceberg Fərqləri
| Xüsusiyyət | Delta | Iceberg |
| Time Travel | `versionAsOf` | `VERSION AS OF snapshot_id` |
| History | `delta_table.history()` | `.snapshots` metadata |
| MERGE | Python API + SQL | Yalnız SQL |
| Catalog | `spark_catalog` | Ayrıca catalog konfiqurasiyası |

### 4. Qarşılaşdığım Problemlər
- **PATH_NOT_FOUND**: CSV path `/home/jovyan/work/` deyil, `/home/jovyan/` idi
- **DELTA_FAILED_TO_MERGE_FIELDS**: `id` sütunu LongType/IntegerType uyğunsuzluğu — `.cast(IntegerType())` ilə həll edildi
- **Iceberg Time Travel**: `format("iceberg").load()` işləmədi — SQL `VERSION AS OF` ilə həll edildi

In [34]:
# Hər müştərinin transaction sayını hesabla
customer_counts = (
    spark.read.format("delta").load(DELTA_SILVER)
    .groupBy("customer_id")
    .agg(F.count("id").alias("tx_count"))
)

customer_loyalty = customer_counts.withColumn(
    "loyalty_score",
    F.when(F.col("tx_count") >= 58, "GOLD")
     .when(F.col("tx_count") >= 48, "SILVER")
     .otherwise("BRONZE")
)

customer_loyalty.groupBy("loyalty_score").count().show()

# Silver cədvəlinə qoş
silver_with_loyalty = (
    spark.read.format("delta").load(DELTA_SILVER)
    .join(customer_loyalty.select("customer_id", "loyalty_score"), on="customer_id", how="left")
)

silver_with_loyalty.select("id", "customer_id", "amount", "loyalty_score").show(10)

+-------------+-----+
|loyalty_score|count|
+-------------+-----+
|       SILVER|  522|
|         GOLD|  148|
|       BRONZE|  330|
+-------------+-----+

+---+-----------+------+-------------+
| id|customer_id|amount|loyalty_score|
+---+-----------+------+-------------+
|  1|        642|584.01|         GOLD|
|  2|        934|215.26|         GOLD|
|  3|        711|124.14|       SILVER|
|  4|        574|322.37|       BRONZE|
|  5|        596|210.36|       BRONZE|
|  6|        572|280.32|       BRONZE|
|  7|        308|423.78|       SILVER|
|  8|        670|194.36|       BRONZE|
|  9|        147| 164.8|       BRONZE|
| 10|        672| 85.95|       BRONZE|
+---+-----------+------+-------------+
only showing top 10 rows



In [35]:
# Silver cədvəlinə qoş
silver_with_loyalty = (
    spark.read.format("delta").load(DELTA_SILVER)
    .join(customer_loyalty.select("customer_id", "loyalty_score"), on="customer_id", how="left")
)

# Delta-ya yaz (mergeSchema — yeni sütunu qəbul edir)
(silver_with_loyalty.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_SILVER)
)

print("Delta Schema Evolution tamamlandı")
print("Yeni schema:")
spark.read.format("delta").load(DELTA_SILVER).printSchema()
spark.read.format("delta").load(DELTA_SILVER).select("id", "customer_id", "amount", "loyalty_score").show(10)

Delta Schema Evolution tamamlandı
Yeni schema:
root
 |-- customer_id: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- dt: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- loyalty_score: string (nullable = true)

+---+-----------+------+-------------+
| id|customer_id|amount|loyalty_score|
+---+-----------+------+-------------+
|  1|        642|584.01|         GOLD|
|  2|        934|215.26|         GOLD|
|  3|        711|124.14|       SILVER|
|  4|        574|322.37|       BRONZE|
|  5|        596|210.36|       BRONZE|
|  6|        572|280.32|       BRONZE|
|  7|        308|423.78|       SILVER|
|  8|        670|194.36|       BRONZE|
|  9|        147| 164.8|       BRONZE|
| 10|        672| 85.95|       BRONZE|
+---+-----------+------+-------------+
only showing top 10 rows



In [36]:
# Iceberg Schema Evolution
print("Əvvəlki Iceberg schema:")
spark.table(ICEBERG_SILVER).printSchema()

# Yeni sütun əlavə et
spark.sql(f"ALTER TABLE {ICEBERG_SILVER} ADD COLUMN loyalty_score STRING")

# Loyalty score yenilə
spark.sql(f"""
    MERGE INTO {ICEBERG_SILVER} AS t
    USING (SELECT customer_id, loyalty_score FROM (
        SELECT customer_id, 
        CASE 
            WHEN tx_count >= 58 THEN 'GOLD'
            WHEN tx_count >= 48 THEN 'SILVER'
            ELSE 'BRONZE'
        END AS loyalty_score
        FROM (
            SELECT customer_id, COUNT(id) as tx_count 
            FROM {ICEBERG_SILVER} 
            GROUP BY customer_id
        )
    )) AS s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET t.loyalty_score = s.loyalty_score
""")

print("Iceberg Schema Evolution tamamlandı")
spark.table(ICEBERG_SILVER).printSchema()
spark.table(ICEBERG_SILVER).select("id", "customer_id", "amount", "loyalty_score").show(10)

Əvvəlki Iceberg schema:
root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- dt: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)

Iceberg Schema Evolution tamamlandı
root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- dt: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- loyalty_score: string (nullable = true)

+---+-----------+------+-------------+
| id|customer_id|amount|loyalty_score|
+---+-----------+------+-------------+
|  1|        642|584.01|         GOLD|
|  2|        934|215.26|         GOLD|
|  3|        711|124.14|       SILVER|
|  4|        574|322.37|       BRONZE|
|  5|        596|210.36|       BRONZE|
|  6|        572|280.32|       BRONZE|
|  7|        308|423.78|       SILVER|
|  8|        670|194.36|       BRONZE|


## Schema Evolution Müqayisəsi: Delta vs Iceberg

| Xüsusiyyət | Delta | Iceberg -|
| Sütun əlavə etmə | `ALTER TABLE delta.\`path\` ADD COLUMN` | `ALTER TABLE iceberg.db.table ADD COLUMN` |
| Mövcud data | NULL olaraq qalır | NULL olaraq qalır |
| Sütunu doldurma | Python API ilə `update()` | SQL `MERGE INTO` ilə |
| Schema saxlama | `overwriteSchema=true` lazımdır | Avtomatik idarə olunur |
| Çətinlik | Path-based olduğu üçün syntax uzundur | Catalog-based, SQL ilə sadədir |

### Nəticə
- **Delta**: Schema dəyişikliyi üçün `overwriteSchema` option-u əlavə etmək lazımdır, əks halda xəta verir
- **Iceberg**: `ALTER TABLE` ilə sütun əlavə etmək daha sadədir, catalog avtomatik idarə edir
- **Hər ikisində**: Mövcud data itmir, yeni sütun `NULL` olaraq əlavə olunur və sonradan doldurulur